In [ ]:
!pip install -q transformers accelerate sentencepiece langdetect protobuf
!pip install -q google-genai bitsandbytes
!pip install -q trl peft datasets rouge-score
!pip install -q spacy scikit-learn nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [27]:

# ============================================================
# -- Step 2 : Environment detection + GPU setup
# ============================================================
import torch, os, re, time, collections, random, functools

# ── 2a : Detect runtime environment ──────────────────────────
IS_COLAB  = 'google.colab' in str(__import__('sys').modules)
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

if IS_COLAB:   print("Environment : Google Colab")
elif IS_KAGGLE: print("Environment : Kaggle")
else:           print("Environment : Local / other")

# ── 2b : GPU detection ───────────────────────────────────────
IS_GPU = torch.cuda.is_available()
NUM_CPU_CORES = os.cpu_count() or 2

if IS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  : {gpu_name} ({vram_gb:.1f} GB VRAM)")
    IS_A100 = "A100" in gpu_name
    IS_T4   = "T4"   in gpu_name
else:
    print("No GPU -- models run on CPU (slow but functional)")
    IS_A100 = IS_T4 = False

device = "cuda" if IS_GPU else "cpu"
print(f"Device : {device}")
print()
print("Expected VRAM (GPU mode):")
print("  mT5-Large  (float16) : ~2.5 GB")
print("  Sarvam-2B  (float16) : ~5.0 GB")
print("  Gemma-2-2B (4-bit)   : ~2.0 GB")
print("  Total (all loaded)   : ~9.5 GB  -- load selectively on T4 16GB")

# ── 2c : NLP libraries ───────────────────────────────────────
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
from nltk.corpus import stopwords as nltk_stopwords
STOPWORDS_EN = set(nltk_stopwords.words('english'))

import spacy
nlp_spacy = spacy.load("en_core_web_sm")   # NER + POS + dependency parse

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
ROUGE = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

print("NLP libraries loaded: spaCy, NLTK, scikit-learn TF-IDF, ROUGE-L")

Environment : Google Colab
GPU  : Tesla T4 (15.6 GB VRAM)
Device : cuda

Expected VRAM (GPU mode):
  mT5-Large  (float16) : ~2.5 GB
  Sarvam-2B  (float16) : ~5.0 GB
  Gemma-2-2B (4-bit)   : ~2.0 GB
  Total (all loaded)   : ~9.5 GB  -- load selectively on T4 16GB
NLP libraries loaded: spaCy, NLTK, scikit-learn TF-IDF, ROUGE-L


In [3]:
# ============================================================
# -- Step 3 : Storage setup (Colab Drive OR local folder)
# ============================================================
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/chatbot_v13"
        print(f"Google Drive mounted. Save root: {SAVE_ROOT}")
    except Exception as e:
        print(f"Drive mount failed ({e}) -- saving to /content/chatbot_v13")
        SAVE_ROOT = "/content/chatbot_v13"
elif IS_KAGGLE:
    SAVE_ROOT = "/kaggle/working/chatbot_v13"
else:
    SAVE_ROOT = os.path.join(os.getcwd(), "chatbot_v13_output")

os.makedirs(SAVE_ROOT, exist_ok=True)
print(f"Output folder: {SAVE_ROOT}")


Drive mount failed (Error: credential propagation was unsuccessful) -- saving to /content/chatbot_v13
Output folder: /content/chatbot_v13


In [4]:
# ============================================================
# -- Step 4 : Load Sarvam-2B (Indian language specialist)
# ============================================================
# NLP concept: subword tokenization (SentencePiece), multilingual pre-training
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

SARVAM_MODEL_NAME = "sarvamai/sarvam-2b-v0.5"
print(f"\nLoading {SARVAM_MODEL_NAME} ...")

try:
    sarvam_tokenizer = AutoTokenizer.from_pretrained(SARVAM_MODEL_NAME)
    sarvam_model = AutoModelForCausalLM.from_pretrained(
        SARVAM_MODEL_NAME,
        torch_dtype       = torch.float16 if IS_GPU else torch.float32,
        device_map        = "auto",
        low_cpu_mem_usage = True,
    )
    sarvam_model.eval()
    if sarvam_tokenizer.pad_token is None:
        sarvam_tokenizer.pad_token = sarvam_tokenizer.eos_token
    n_params = sum(p.numel() for p in sarvam_model.parameters()) / 1e9
    print(f"Sarvam-2B loaded ({n_params:.1f}B params)")
    SARVAM_READY = True
except Exception as e:
    print(f"Sarvam failed to load: {e}")
    SARVAM_READY = False



Loading sarvamai/sarvam-2b-v0.5 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Sarvam-2B loaded (2.5B params)


In [5]:
# ============================================================
# -- Step 5 : Load Gemma-2-2B-IT (English reasoning specialist)
# ============================================================
# NLP concept: instruction tuning, 4-bit quantization, causal LM
from transformers import BitsAndBytesConfig

GEMMA_MODEL_NAME = "google/gemma-2-2b-it"
print(f"\nLoading {GEMMA_MODEL_NAME} ...")

# CPU-safe loading: use int8 on CPU (needs no CUDA), 4-bit on GPU
if IS_GPU:
    gemma_load_kwargs = dict(
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        ),
        torch_dtype = torch.float16,
    )
else:
    gemma_load_kwargs = dict(torch_dtype=torch.float32)   # CPU: full precision, slower

try:
    gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_NAME)
    gemma_model = AutoModelForCausalLM.from_pretrained(
        GEMMA_MODEL_NAME,
        **gemma_load_kwargs,
        device_map        = "auto",
        low_cpu_mem_usage = True,
    )
    gemma_model.eval()
    if gemma_tokenizer.pad_token is None:
        gemma_tokenizer.pad_token = gemma_tokenizer.eos_token
    n_params = sum(p.numel() for p in gemma_model.parameters()) / 1e9
    print(f"Gemma-2-2B-IT loaded ({n_params:.1f}B params)")
    GEMMA_READY = True
except Exception as e:
    print(f"Gemma failed to load: {e}"); GEMMA_READY = False

# Backward-compatible aliases (keep Step 6c working unchanged)
llama_model     = gemma_model     if GEMMA_READY else None
llama_tokenizer = gemma_tokenizer if GEMMA_READY else None
LLAMA_READY     = GEMMA_READY



Loading google/gemma-2-2b-it ...
Gemma failed to load: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2-2b-it.
401 Client Error. (Request ID: Root=1-6a3fc9b5-5e13ee6d3e92b68b4bc621aa;c7ec9c31-5c65-42cc-b37b-ce13ebb12e9f)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.


### Set up Gemini API

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio. In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`.

In [81]:
import google.generativeai as genai
from google.colab import userdata

print('\nLoading Gemini ...')

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Using the 'latest' alias which was confirmed available in the model list
    GEMINI_MODEL = 'models/gemini-flash-latest'
    client = genai.GenerativeModel(model_name=GEMINI_MODEL)

    GEMINI_READY = True
    print(f'✓ Gemini model ({GEMINI_MODEL}) loaded.')
except Exception as e:
    print(f'✗ Gemini failed to load: {e}')
    GEMINI_READY = False


Loading Gemini ...
✓ Gemini model (models/gemini-flash-latest) loaded.


In [69]:
# List available models to find the correct Gemini string
print("Listing available models for the current API key:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
except Exception as e:
    print(f"Error listing models: {e}")

Listing available models for the current API key:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.5-flash
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flash-tts-preview

In [68]:
# List available models to find the correct Gemini string
print("Listing available models for the current API key:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
except Exception as e:
    print(f"Error listing models: {e}")

Listing available models for the current API key:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.5-flash
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/gemini-3.1-flash-tts-preview

In [30]:
# ============================================================
# STEP 5B: MODULAR BREAKDOWN
# ============================================================

# ============================================================
# STEP 5b-A : Load mT5-Large Model
# ============================================================
print("\n" + "="*62)
print("STEP 5b-A: Load mT5-Large Model")
print("="*62)

from transformers import MT5ForConditionalGeneration, T5Tokenizer
import torch

MT5_MODEL_NAME       = "google/mt5-large"
MT5_SCORE_THRESHOLD  = 0.30
MT5_MAX_INPUT_LEN    = 256
MT5_MAX_TARGET_LEN   = 200
MT5_BEAM_SIZE        = 4

print(f"\nLoading {MT5_MODEL_NAME} ...")
print("(First time: downloads ~2.5GB)")

try:
    mt5_tokenizer = T5Tokenizer.from_pretrained(MT5_MODEL_NAME)
    mt5_model = MT5ForConditionalGeneration.from_pretrained(
        MT5_MODEL_NAME,
        torch_dtype       = torch.float16 if IS_GPU else torch.float32,
        device_map        = "auto",
        low_cpu_mem_usage = True,
    )
    mt5_model.eval()
    n_params = sum(p.numel() for p in mt5_model.parameters()) / 1e9
    print(f"✓ mT5-Large loaded ({n_params:.1f}B params)")
    print(f"  Device: {device}")
    print(f"  Tokenizer vocab size: {mt5_tokenizer.vocab_size}")
    MT5_READY = True
except Exception as e:
    print(f"✗ mT5-Large failed to load: {e}")
    MT5_READY = False

if not MT5_READY:
    print("\n⚠️  Cannot proceed without mT5. Check GPU memory and internet connection.")
    print("   Stopping here.")
else:
    print("\n✓ Step 5b-A complete. Proceed to Step 5b-B.\n")


STEP 5b-A: Load mT5-Large Model

Loading google/mt5-large ...
(First time: downloads ~2.5GB)


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✓ mT5-Large loaded (1.2B params)
  Device: cuda
  Tokenizer vocab size: 250100

✓ Step 5b-A complete. Proceed to Step 5b-B.



### Adjusting mT5 Score Threshold

To observe the backfill models in action, we are increasing the `MT5_SCORE_THRESHOLD` to `0.9`. This will make the primary mT5 model's responses less likely to meet the threshold, causing the system to rely more on the backfill models (Sarvam, Gemma, Gemini).

In [46]:
# Adjust MT5_SCORE_THRESHOLD to a higher value to trigger backfill models more often
MT5_SCORE_THRESHOLD = 0.9
print(f"MT5_SCORE_THRESHOLD adjusted to: {MT5_SCORE_THRESHOLD}")

MT5_SCORE_THRESHOLD adjusted to: 0.9


In [31]:
# ============================================================
# STEP 5b-B : Define Multilingual Dataset
# ============================================================
print("="*62)
print("STEP 5b-B: Define Multilingual Dataset (FINETUNE_DATA)")
print("="*62)

FINETUNE_DATA = [
    # English (2 examples)
    {"instruction": "What is machine learning?",
     "response": "Machine learning is a subset of AI where systems learn patterns from data to make predictions without explicit programming."},
    {"instruction": "What is a transformer model?",
     "response": "A transformer uses self-attention to process sequences in parallel, making it highly effective for NLP tasks."},

    # Hindi (2 examples)
    {"instruction": "मशीन लर्निंग क्या है?",
     "response": "मशीन लर्निंग एक AI तकनीक है जिसमें कंप्यूटर डेटा से सीखता है और बिना स्पष्ट प्रोग्रामिंग के निर्णय लेता है।"},
    {"instruction": "ट्रांसफॉर्मर मॉडल क्या है?",
     "response": "ट्रांसफॉर्मर एक तंत्रिका नेटवर्क आर्किटेक्चर है जो स्व-ध्यान तंत्र का उपयोग करके अनुक्रम डेटा को प्रक्रिया करता है।"},

    # Kannada (2 examples)
    {"instruction": "ಕಂಪ್ಯೂಟರ್ ಎಂದರೇನು?",
     "response": "ಕಂಪ್ಯೂಟರ್ ಒಂದು ಎಲೆಕ್ಟ್ರಾನಿಕ್ ಸಾಧನವಾಗಿದ್ದು, ಇದು ಡೇಟಾವನ್ನು ಪ್ರಕ್ರಿಯೆಗೊಳಿಸುತ್ತದೆ."},
    {"instruction": "ಆರ್ಟಿಫಿಶಿಯಲ್ ಇಂಟೆಲಿಜೆನ್ಸ್ ಎಂದರೇನು?",
     "response": "ಆರ್ಟಿಫಿಶಿಯಲ್ ಇಂಟೆಲಿಜೆನ್ಸ್ ಎನ್ನುವುದು ಮನುಷ್ಯನ ಸಮಾನ ಬುದ್ಧಿಮತ್ತೆಯನ್ನು ಪ್ರದರ್ಶಿಸುವ ಕಂಪ್ಯೂಟರ್ ಸಿಸ್ಟಮ್."},

    # Hinglish (2 examples)
    {"instruction": "Machine learning kya hota hai?",
     "response": "Machine learning ek AI technique hai jisme computer data se seekhta hai aur khud decisions leta hai."},
    {"instruction": "Neural network kaise kaam karta hai?",
     "response": "Neural network human brain ki tarah kaam karta hai aur patterns ko identify karta hai."},
]

print(f"\n✓ Dataset defined: {len(FINETUNE_DATA)} samples")
for i, sample in enumerate(FINETUNE_DATA, 1):
    lang_label = "EN" if ord(sample["instruction"][0]) < 128 else "LANG"
    instr_preview = sample["instruction"][:40] + ("..." if len(sample["instruction"]) > 40 else "")
    print(f"  [{i}] {lang_label}: {instr_preview}")

print("\nSample distribution:")
print(f"  English   : 2")
print(f"  Hindi     : 2")
print(f"  Kannada   : 2")
print(f"  Hinglish  : 2")
print(f"  Total     : {len(FINETUNE_DATA)}")

print("\n✓ Step 5b-B complete. Proceed to Step 5b-C.\n")

STEP 5b-B: Define Multilingual Dataset (FINETUNE_DATA)

✓ Dataset defined: 8 samples
  [1] EN: What is machine learning?
  [2] EN: What is a transformer model?
  [3] LANG: मशीन लर्निंग क्या है?
  [4] LANG: ट्रांसफॉर्मर मॉडल क्या है?
  [5] LANG: ಕಂಪ್ಯೂಟರ್ ಎಂದರೇನು?
  [6] LANG: ಆರ್ಟಿಫಿಶಿಯಲ್ ಇಂಟೆಲಿಜೆನ್ಸ್ ಎಂದರೇನು?
  [7] EN: Machine learning kya hota hai?
  [8] EN: Neural network kaise kaam karta hai?

Sample distribution:
  English   : 2
  Hindi     : 2
  Kannada   : 2
  Hinglish  : 2
  Total     : 8

✓ Step 5b-B complete. Proceed to Step 5b-C.



In [32]:
# ============================================================
# STEP 5b-C : Define Text Normalization + Preprocessing
# ============================================================
print("="*62)
print("STEP 5b-C: Define Preprocessing Function (text_target FIX)")
print("="*62)

def normalize_text(text):
    """NLP: text normalization -- lowercase, strip extra whitespace."""
    return re.sub(r'\s+', ' ', text.strip())

def preprocess_mt5(sample):
    """
    Tokenize instruction (input) and response (target) for mT5 seq2seq.

    KEY FIX: Uses text_target= parameter instead of deprecated as_target_tokenizer()

    Returns: dict with input_ids, attention_mask, and labels (for seq2seq loss)
    """
    input_text  = "answer: " + normalize_text(sample["instruction"])
    target_text = normalize_text(sample["response"])

    # Step 1: Tokenize input (encoder input)
    model_inputs = mt5_tokenizer(
        input_text,
        max_length=MT5_MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    # Step 2: Tokenize target using text_target= (MODERN APPROACH) ✅
    # This is the FIXED version that replaces as_target_tokenizer()
    labels = mt5_tokenizer(
        text_target=target_text,  # ← KEY FIX: text_target= parameter
        max_length=MT5_MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )

    # Step 3: Mask padding tokens in labels (loss ignores -100)
    # NLP concept: padding tokens should not contribute to loss
    label_ids = [
        (l if l != mt5_tokenizer.pad_token_id else -100)
        for l in labels["input_ids"]
    ]

    model_inputs["labels"] = label_ids
    return model_inputs

# Test the function on first sample
print("\n🔍 Testing preprocess_mt5() on first sample...")
test_sample = FINETUNE_DATA[0]
print(f"Input: {test_sample['instruction'][:50]}...")

try:
    processed = preprocess_mt5(test_sample)
    print(f"✓ Preprocessing works!")
    print(f"  Input tokens   : {len(processed['input_ids'])} (max: {MT5_MAX_INPUT_LEN})")
    print(f"  Label tokens   : {len(processed['labels'])} (max: {MT5_MAX_TARGET_LEN})")
    print(f"  Attention mask : {len(processed['attention_mask'])}")
    print(f"  Sample input_ids (first 10): {processed['input_ids'][:10]}")
    print(f"  Sample labels (first 10)   : {processed['labels'][:10]}")
except Exception as e:
    print(f"✗ Preprocessing failed: {e}")
    import traceback
    traceback.print_exc()

print("\n✓ Step 5b-C complete. Proceed to Step 5b-D.\n")




STEP 5b-C: Define Preprocessing Function (text_target FIX)

🔍 Testing preprocess_mt5() on first sample...
Input: What is machine learning?...
✓ Preprocessing works!
  Input tokens   : 256 (max: 256)
  Label tokens   : 200 (max: 200)
  Attention mask : 256
  Sample input_ids (first 10): [20954, 267, 5126, 339, 10902, 22651, 291, 1, 0, 0]
  Sample labels (first 10)   : [10755, 22651, 339, 259, 262, 2411, 2325, 304, 32419, 259]

✓ Step 5b-C complete. Proceed to Step 5b-D.



In [33]:
# ============================================================
# STEP 5b-D : Create Train/Val Split + Dataset Objects
# ============================================================
print("="*62)
print("STEP 5b-D: Create Train/Val Split")
print("="*62)

from datasets import Dataset
import random

if not MT5_READY:
    print("\n⚠️  MT5 not loaded. Skipping dataset split.")
else:
    # Shuffle and split
    random.seed(42)
    shuffled = FINETUNE_DATA[:]
    random.shuffle(shuffled)

    val_size   = max(1, int(len(shuffled) * 0.2))  # 20% validation
    val_data   = shuffled[:val_size]
    train_data = shuffled[val_size:]

    print(f"\n📊 Dataset split:")
    print(f"  Total    : {len(shuffled)}")
    print(f"  Train    : {len(train_data)} ({len(train_data)/len(shuffled)*100:.0f}%)")
    print(f"  Val      : {len(val_data)} ({len(val_data)/len(shuffled)*100:.0f}%)")

    # Create HuggingFace Dataset objects
    print(f"\n🔄 Creating HuggingFace Dataset objects...")
    raw_train = Dataset.from_list(train_data)
    raw_val   = Dataset.from_list(val_data)
    print(f"✓ Raw datasets created")

    # Preprocess datasets
    print(f"\n🔄 Preprocessing train dataset ({len(raw_train)} samples)...")
    try:
        train_dataset = raw_train.map(
            preprocess_mt5,
            remove_columns=raw_train.column_names,
            desc="Preprocessing train"
        )
        print(f"✓ Train dataset preprocessed")
        print(f"  Features: {train_dataset.column_names}")
    except Exception as e:
        print(f"✗ Train preprocessing failed: {e}")
        import traceback
        traceback.print_exc()
        train_dataset = None

    print(f"\n🔄 Preprocessing val dataset ({len(raw_val)} samples)...")
    try:
        val_dataset = raw_val.map(
            preprocess_mt5,
            remove_columns=raw_val.column_names,
            desc="Preprocessing val"
        )
        print(f"✓ Val dataset preprocessed")
        print(f"  Features: {val_dataset.column_names}")
    except Exception as e:
        print(f"✗ Val preprocessing failed: {e}")
        import traceback
        traceback.print_exc()
        val_dataset = None

    if train_dataset and val_dataset:
        print("\n✓ Step 5b-D complete. Proceed to Step 5b-E.\n")
    else:
        print("\n⚠️  Preprocessing failed. Check errors above.")

STEP 5b-D: Create Train/Val Split

📊 Dataset split:
  Total    : 8
  Train    : 7 (88%)
  Val      : 1 (12%)

🔄 Creating HuggingFace Dataset objects...
✓ Raw datasets created

🔄 Preprocessing train dataset (7 samples)...


Preprocessing train:   0%|          | 0/7 [00:00<?, ? examples/s]

✓ Train dataset preprocessed
  Features: ['input_ids', 'attention_mask', 'labels']

🔄 Preprocessing val dataset (1 samples)...


Preprocessing val:   0%|          | 0/1 [00:00<?, ? examples/s]

✓ Val dataset preprocessed
  Features: ['input_ids', 'attention_mask', 'labels']

✓ Step 5b-D complete. Proceed to Step 5b-E.



In [34]:
# ============================================================
# STEP 5b-E : Define ROUGE-L Evaluation Metric
# ============================================================
print("="*62)
print("STEP 5b-E: Define ROUGE-L Evaluation Metric")
print("="*62)

import numpy as np
from rouge_score import rouge_scorer

# Initialize ROUGE scorer (already loaded in Step 2c, but re-define here for clarity)
ROUGE = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_rouge(eval_pred):
    """
    Compute ROUGE-L score for seq2seq evaluation.

    Called by trainer to evaluate on validation set.
    Returns dict with rougeL metric value.
    """
    preds, labels = eval_pred

    # Handle tuple output (if trainer returns multiple predictions)
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 (masked padding) back to pad_token_id for decoding
    preds  = np.where(preds  != -100, preds,  mt5_tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, mt5_tokenizer.pad_token_id)

    # Decode token IDs to text
    decoded_preds  = mt5_tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = mt5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE-L for each pred-label pair
    scores = [
        ROUGE.score(ref, hyp)['rougeL'].fmeasure
        for ref, hyp in zip(decoded_labels, decoded_preds)
    ]

    return {"rougeL": round(float(np.mean(scores)), 4)}

# Test on a dummy batch
print("\n🔍 Testing compute_rouge() on dummy batch...")
try:
    dummy_preds = np.array([[1, 2, 3, -100], [4, 5, 6, -100]])
    dummy_labels = np.array([[1, 2, 3, -100], [4, 5, 6, -100]])
    test_output = compute_rouge((dummy_preds, dummy_labels))
    print(f"✓ compute_rouge() works!")
    print(f"  Sample output: {test_output}")
except Exception as e:
    print(f"✗ compute_rouge() failed: {e}")
    import traceback
    traceback.print_exc()

print("\n✓ Step 5b-E complete. Proceed to Step 5b-F.\n")

STEP 5b-E: Define ROUGE-L Evaluation Metric

🔍 Testing compute_rouge() on dummy batch...
✓ compute_rouge() works!
  Sample output: {'rougeL': 1.0}

✓ Step 5b-E complete. Proceed to Step 5b-F.



In [35]:
# ============================================================
# STEP 5b-F : Configure LoRA for Parameter-Efficient Fine-Tuning
# ============================================================
print("="*62)
print("STEP 5b-F: Configure LoRA for mT5-Large")
print("="*62)

if not IS_GPU:
    print("\n⚠️  No GPU detected. LoRA training requires GPU.")
    print("   Skipping LoRA config. (CPU training would be extremely slow)")
    LORA_READY = False
else:
    from peft import LoraConfig, get_peft_model, TaskType

    print("\n🔧 Configuring LoRA...")
    lora_config = LoraConfig(
        r              = 16,              # LoRA rank
        lora_alpha     = 32,              # LoRA scaling
        target_modules = [
            "q", "k", "v", "o",          # Attention layers
            "wi_0", "wi_1", "wo"         # Encoder + decoder FFN
        ],
        lora_dropout   = 0.05,
        bias           = "none",
        task_type      = TaskType.SEQ_2_SEQ_LM,  # Seq2seq task
    )
    print(f"✓ LoRA config defined:")
    print(f"  Rank           : {lora_config.r}")
    print(f"  Alpha          : {lora_config.lora_alpha}")
    print(f"  Dropout        : {lora_config.lora_dropout}")
    print(f"  Target modules : {lora_config.target_modules}")

    # Apply LoRA to model
    print(f"\n🔄 Applying LoRA to mT5-Large...")
    try:
        mt5_model.train()
        mt5_model = get_peft_model(mt5_model, lora_config)
        mt5_model.print_trainable_parameters()

        # Enable gradient checkpointing to save VRAM
        mt5_model.gradient_checkpointing_enable()
        print(f"✓ Gradient checkpointing enabled")

        LORA_READY = True
        print(f"\n✓ Step 5b-F complete. Proceed to Step 5b-G.\n")
    except Exception as e:
        print(f"✗ LoRA setup failed: {e}")
        import traceback
        traceback.print_exc()
        LORA_READY = False


STEP 5b-F: Configure LoRA for mT5-Large

🔧 Configuring LoRA...
✓ LoRA config defined:
  Rank           : 16
  Alpha          : 32
  Dropout        : 0.05
  Target modules : {'q', 'v', 'o', 'k', 'wo', 'wi_1', 'wi_0'}

🔄 Applying LoRA to mT5-Large...
trainable params: 18,284,544 || all params: 1,247,865,856 || trainable%: 1.4653
✓ Gradient checkpointing enabled

✓ Step 5b-F complete. Proceed to Step 5b-G.



In [36]:
# ============================================================
# STEP 5b-G : Training Loop (LoRA Fine-tuning) - FIXED
# ============================================================
print("="*62)
print("STEP 5b-G: Train mT5-Large with LoRA")
print("="*62)

from transformers import (
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)

if not LORA_READY or not MT5_READY:
    print("\n⚠️  Cannot train: LoRA not ready or mT5 not loaded.")
    print("   Check previous steps for errors.")
elif train_dataset is None or val_dataset is None:
    print("\n⚠️  Cannot train: datasets not created.")
    print("   Check Step 5b-D for errors.")
else:
    print("\n📋 Training configuration:")
    print(f"  Epochs                   : 5")
    print(f"  Batch size               : 2")
    print(f"  Gradient accumulation    : 4")
    print(f"  Learning rate            : 3e-4")
    print(f"  LR scheduler             : cosine")
    print(f"  Max input length         : {MT5_MAX_INPUT_LEN}")
    print(f"  Max target length        : {MT5_MAX_TARGET_LEN}")
    print(f"  Beam size (generation)   : {MT5_BEAM_SIZE}")
    print(f"  Early stopping patience  : 2")

    # Create data collator
    data_collator = DataCollatorForSeq2Seq(
        mt5_tokenizer, model=mt5_model,
        label_pad_token_id=-100, pad_to_multiple_of=8
    )

    # Calculate warmup steps (replacing deprecated warmup_ratio)
    total_steps = (len(train_dataset) // 2) * 5  # batch_size=2, epochs=5
    warmup_steps = int(total_steps * 0.1)  # 10% warmup

    # Training arguments (FIXED: removed tokenizer, use warmup_steps)
    training_args = Seq2SeqTrainingArguments(
        output_dir                  = os.path.join(SAVE_ROOT, "mt5_finetuned"),
        num_train_epochs            = 5,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 2,
        gradient_accumulation_steps = 4,
        learning_rate               = 3e-4,
        fp16                        = IS_GPU,
        predict_with_generate       = True,
        generation_max_length       = MT5_MAX_TARGET_LEN,
        generation_num_beams        = MT5_BEAM_SIZE,
        logging_steps               = 5,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "rougeL",
        greater_is_better           = True,
        warmup_steps                = warmup_steps,  # ← FIX: use warmup_steps instead of warmup_ratio
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
    )

    # Initialize trainer (FIXED: removed tokenizer parameter)
    trainer = Seq2SeqTrainer(
        model          = mt5_model,
        args           = training_args,
        train_dataset  = train_dataset,
        eval_dataset   = val_dataset,
        # tokenizer    = mt5_tokenizer,  # ← REMOVED: not supported in newer transformers
        data_collator  = data_collator,
        compute_metrics= compute_rouge,
        callbacks      = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print(f"\n▶️  Starting training...")
    print(f"   Output dir: {training_args.output_dir}")
    print(f"   Warmup steps: {warmup_steps}")

    try:
        trainer.train()
        print(f"\n✓ Training complete!")

        # Merge LoRA weights and save
        print(f"\n💾 Saving model...")
        mt5_model = mt5_model.merge_and_unload()
        mt5_model.eval()

        save_path = os.path.join(SAVE_ROOT, "mt5_final")
        mt5_model.save_pretrained(save_path)
        mt5_tokenizer.save_pretrained(save_path)

        print(f"✓ Model saved to: {save_path}")
        print(f"\n✓ Step 5b-G complete. All training done!")

    except Exception as e:
        print(f"\n✗ Training failed: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*62)
print("STEP 5B: ALL PARTS COMPLETE")
print("="*62)

STEP 5b-G: Train mT5-Large with LoRA

📋 Training configuration:
  Epochs                   : 5
  Batch size               : 2
  Gradient accumulation    : 4
  Learning rate            : 3e-4
  LR scheduler             : cosine
  Max input length         : 256
  Max target length        : 200
  Beam size (generation)   : 4
  Early stopping patience  : 2

▶️  Starting training...
   Output dir: /content/chatbot_v13/mt5_finetuned
   Warmup steps: 1


Epoch,Training Loss,Validation Loss,Rougel
1,No log,25.451672,0.000000
2,No log,25.451672,0.000000
3,No log,25.451672,0.000000



✓ Training complete!

💾 Saving model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to: /content/chatbot_v13/mt5_final

✓ Step 5b-G complete. All training done!

STEP 5B: ALL PARTS COMPLETE


In [37]:
# ============================================================
# -- Step 6 : NLP Pipeline + Language Detection
# ============================================================

# ── Step 6a : Language detection ─────────────────────────────
# NLP concepts: Unicode character range detection, lexicon lookup,
# statistical language ID (langdetect = Naive Bayes + n-grams under the hood)
import re
from langdetect import detect, DetectorFactory, LangDetectException
DetectorFactory.seed = 0

HINDI_WORDS = {
    'ka','ki','ke','hai','hain','kya','aur','nahi','nahin','mein','main',
    'yeh','woh','tum','aap','hum','kahan','kaun','kab','kyun','kaise',
    'bahut','accha','theek','matlab','bolna','karo','bolo','batao',
    'iska','uska','apna','mera','tera','unka','hamara','tumhara','paas',
    'wala','wali','wale','bhi','toh','par','lekin','agar','jab','tab',
    'sab','sirf','abhi','pehle','baad','phir','kuch','kitna','kitni',
    'sahi','galat','hoga','tha','thi','rahega','chahiye','milega','seedha'
}
KANNADA_WORDS = {
    'enu','illa','alli','iga','nanu','neevu','avaru','ivaru','adu','idu',
    'yenu','yaake','hege','yelli','yaavaga','madali','helali','nodali',
    'banni','hogi','barli','bekku','biddi','hoogide','barthini','hogthini'
}
LANG_LABELS  = {
    'hi':'Hindi','kn':'Kannada',
    'mixed_hi':'Hinglish','mixed_kn':'Kannada-English','en':'English'
}
INDIAN_LANGS = {'hi','kn','mixed_hi','mixed_kn'}

def detect_language(text):
    """NLP concept: rule-based + statistical language identification."""
    dev = sum(1 for c in text if '\u0900' <= c <= '\u097F')   # Devanagari
    kan = sum(1 for c in text if '\u0C80' <= c <= '\u0CFF')   # Kannada script
    tot = sum(1 for c in text if c.isalpha())
    if tot == 0:              return 'en'
    if dev / tot > 0.20:     return 'hi'
    if kan / tot > 0.20:     return 'kn'
    words = set(re.findall(r'\b\w+\b', text.lower()))
    km = words & KANNADA_WORDS; hm = words & HINDI_WORDS
    if km and len(km) >= len(hm): return 'mixed_kn'
    if hm:                        return 'mixed_hi'
    try:                          return detect(text)
    except LangDetectException:   return 'en'

print("detect_language() defined.")

# ── Step 6b : NLP feature extraction pipeline ────────────────
# NLP concepts: tokenization, stopword removal, POS tagging, NER,
# TF-IDF vectorization, BM25-style keyword overlap scoring

_tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
_tfidf_corpus = [s["instruction"] + " " + s["response"] for s in FINETUNE_DATA]
try:
    _tfidf.fit(_tfidf_corpus)
    TFIDF_READY = True
except Exception:
    TFIDF_READY = False

def extract_nlp_features(text):
    """
    NLP pipeline: tokenize → remove stopwords → POS tag → NER.
    Returns dict with tokens, keywords, entities, pos_tags.
    """
    doc = nlp_spacy(text[:500])   # spaCy limit for speed
    tokens   = [t.text.lower() for t in doc if not t.is_punct]
    keywords = [t.lemma_.lower() for t in doc
                if not t.is_stop and not t.is_punct and t.pos_ in {'NOUN','VERB','PROPN'}]
    entities = [(ent.text, ent.label_) for ent in doc.ents]   # NER
    pos_tags = [(t.text, t.pos_) for t in doc]                # POS tagging
    return {"tokens": tokens, "keywords": keywords,
            "entities": entities, "pos_tags": pos_tags}

def cosine_tfidf_score(text_a, text_b):
    """NLP concept: TF-IDF cosine similarity between two texts."""
    if not TFIDF_READY:
        return 0.0
    try:
        vecs = _tfidf.transform([text_a, text_b]).toarray()
        return float(cosine_similarity([vecs[0]], [vecs[1]])[0][0])
    except Exception:
        return 0.0

def bm25_keyword_overlap(query, response):
    """
    NLP concept: BM25-style term overlap score (simplified).
    Measures how many query keywords appear in the response.
    """
    q_words = set(re.findall(r'\b\w+\b', query.lower())) - STOPWORDS_EN
    r_words = set(re.findall(r'\b\w+\b', response.lower()))
    if not q_words: return 0.0
    return len(q_words & r_words) / len(q_words)

def ngram_diversity(text, n=2):
    """
    NLP concept: n-gram diversity -- ratio of unique bigrams to total.
    Low diversity = repetitive / looping output.
    """
    tokens = text.lower().split()
    if len(tokens) < n: return 1.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return len(set(ngrams)) / len(ngrams) if ngrams else 1.0

print("NLP feature pipeline defined (TF-IDF, BM25, n-gram diversity, NER, POS).")


detect_language() defined.
NLP feature pipeline defined (TF-IDF, BM25, n-gram diversity, NER, POS).


In [38]:

# ── Step 6c : Response quality scorer ────────────────────────
# NLP concepts: ROUGE-L, BM25 overlap, n-gram diversity, length penalty

RESPONSE_SCORE_WEIGHTS = {
    "length"    : 0.20,   # penalise very short responses
    "diversity" : 0.25,   # penalise repetition
    "keyword"   : 0.30,   # BM25 query-response overlap
    "rouge"     : 0.25,   # ROUGE-L vs a reference (if available)
}

def score_response(query, response, reference=None):
    """
    Composite NLP quality score in [0, 1].
    Used to decide whether mT5 output is good enough or backfill is needed.
    """
    if not response or len(response.strip()) < 5:
        return 0.0

    # 1. Length score (sigmoid-like: 0 for <10 words, 1 for >=50 words)
    word_count  = len(response.split())
    length_sc   = min(word_count / 50.0, 1.0)

    # 2. N-gram diversity
    diversity_sc = ngram_diversity(response, n=2)

    # 3. BM25 keyword overlap
    keyword_sc  = bm25_keyword_overlap(query, response)

    # 4. ROUGE-L vs reference (use first FINETUNE_DATA entry as pseudo-ref if none given)
    if reference:
        rouge_sc = ROUGE.score(reference, response)['rougeL'].fmeasure
    else:
        rouge_sc = 0.5   # neutral if no reference available

    score = (
        RESPONSE_SCORE_WEIGHTS["length"]    * length_sc    +
        RESPONSE_SCORE_WEIGHTS["diversity"] * diversity_sc +
        RESPONSE_SCORE_WEIGHTS["keyword"]   * keyword_sc   +
        RESPONSE_SCORE_WEIGHTS["rouge"]     * rouge_sc
    )
    return round(score, 3)

def is_good_response(response):
    """Legacy guard: basic quality check (used by Sarvam / Gemma / Gemini paths)."""
    if not response or len(response.strip()) < 5:
        return False
    words = response.lower().split()
    if not words: return False
    if len(words) > 5:
        most_common = max(words.count(w) for w in set(words))
        if most_common / len(words) > 0.4:
            return False
    return True

print("score_response() and is_good_response() defined.")

score_response() and is_good_response() defined.


In [39]:
# ── Step 6d : mT5 inference function (PRIMARY model) ─────────
# NLP concepts: seq2seq generation, beam search, length penalty, no-repeat n-gram

MT5_SYSTEM_PREFIX = {
    'en':       "Answer in English: ",
    'hi':       "हिंदी में उत्तर दें: ",
    'kn':       "ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ: ",
    'mixed_hi': "Hinglish mein jawab do: ",
    'mixed_kn': "Kannada-English mix lo answer cheyyi: ",
}

def ask_mt5(text, lang):
    """
    mT5-Large seq2seq inference.
    Returns (response_text, quality_score).
    NLP: beam search decoding, no-repeat n-gram, length penalty.
    """
    if not globals().get("MT5_READY", False):
        return None, 0.0
    try:
        prefix = MT5_SYSTEM_PREFIX.get(lang, "Answer: ")
        input_text = prefix + normalize_text(text)
        inputs = mt5_tokenizer(
            input_text,
            return_tensors  = "pt",
            max_length      = MT5_MAX_INPUT_LEN,
            truncation      = True,
            padding         = True,
        ).to(device)

        with torch.no_grad():
            outputs = mt5_model.generate(
                **inputs,
                max_new_tokens          = MT5_MAX_TARGET_LEN,
                num_beams               = MT5_BEAM_SIZE,      # beam search
                no_repeat_ngram_size    = 3,                  # NLP: no-repeat n-gram
                length_penalty          = 1.2,                # NLP: length penalty
                early_stopping          = True,
                repetition_penalty      = 1.3,                # NLP: repetition penalty
            )

        response = mt5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        score    = score_response(text, response)
        return (response if len(response) > 5 else None), score

    except Exception as e:
        print(f"  [mT5 error: {e}]")
        return None, 0.0

print("ask_mt5() defined.")

ask_mt5() defined.


In [88]:
SARVAM_SYSTEM = (
    "You are a factual assistant. Answer the question directly in 1-2 sentences. "
    "Do not use signatures, timestamps, or forum-style comments. "
    "Respond ONLY in the user's language."
)

def ask_sarvam(text, lang, history=None):
    if not globals().get('SARVAM_READY', False): return None
    try:
        lang_instr = SARVAM_LANG_INSTRUCTIONS.get(lang, '')
        messages = [{'role': 'system', 'content': f'{SARVAM_SYSTEM} {lang_instr}'}]

        # Clear history if it contains hallucinations to prevent model from repeating them
        if history and len(history) > 0:
            last_resp = history[-1].get('content', '')
            if 'Jul 12' in last_resp or 'Deduplicator' in last_resp:
                history.clear()

        messages.append({'role': 'user', 'content': text})
        text_input = sarvam_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = sarvam_tokenizer(text_input, return_tensors='pt', truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = sarvam_model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                repetition_penalty=1.5,
                pad_token_id=sarvam_tokenizer.eos_token_id
            )

        response = sarvam_tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()

        # --- HALLUCINATION FILTER ---
        # Specifically targeting the 'Deduplicator/Jul 12' forum-scrape pattern
        blacklist = ['Jul 12', 'Deduplicator', 'at 0:37', 'printf(', 'strcpy(', 'compiler']
        if any(item.lower() in response.lower() for item in blacklist) or len(response) < 5:
            return None # Trigger fallback to Gemini

        return response
    except Exception as e:
        print(f'  [Sarvam error: {e}]')
        return None

In [41]:

# ── Step 6f : Gemma response function ────────────────────────
LLAMA_SYSTEM = (
    "You are a helpful, accurate, and concise assistant. "
    "Answer any question on any topic clearly. "
    "Always reply in English. "
    "Keep answers focused and under 200 words unless more detail is needed."
)

def ask_llama(text, lang, history=None):
    if not globals().get("LLAMA_READY", False): return None
    try:
        messages = [{"role":"system","content":LLAMA_SYSTEM}]
        if history:
            for turn in history[-4:]:
                messages.append({"role":turn["role"],"content":turn["content"]})
        messages.append({"role":"user","content":text})
        text_input = llama_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = llama_tokenizer(
            text_input, return_tensors="pt", truncation=True, max_length=1024
        ).to(device)
        with torch.no_grad():
            outputs = llama_model.generate(
                **inputs,
                max_new_tokens     = 300,
                temperature        = 0.7,
                do_sample          = True,
                pad_token_id       = llama_tokenizer.eos_token_id,
                repetition_penalty = 1.1,
            )
        response = llama_tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
        return response if len(response) > 3 else None
    except Exception as e:
        print(f"  [Gemma error: {e}]"); return None

print("ask_llama() defined.")

ask_llama() defined.


In [57]:
def ask_gemini(text, lang, history=None):
    _client = globals().get("client")
    _ready  = globals().get("GEMINI_READY", False)
    if not _ready or not _client: return None
    try:
        lang_instr = GEMINI_LANG_INSTRUCTIONS.get(lang, "")
        messages_for_api = []

        if not history:
            messages_for_api.append({"role": "user", "parts": [{"text": GEMINI_SYSTEM}]})
            messages_for_api.append({"role": "user", "parts": [{"text": f"{lang_instr}\n\nUser: {text}"}]})
        else:
            for turn in history:
                messages_for_api.append({"role": turn["role"], "parts": turn["parts"]})
            messages_for_api.append({"role": "user", "parts": [{"text": f"{lang_instr}\n\nUser: {text}"}]})

        resp = _client.generate_content(
            contents=messages_for_api,
            generation_config=genai.GenerationConfig(max_output_tokens=512)
        )
        reply = resp.text.strip()
        return reply if len(reply) > 3 else None
    except Exception as e:
        print(f"  [Gemini error: {e}]"); return None

In [43]:
# ============================================================
# -- Step 7 : NLP-enhanced response scoring display
# ============================================================
# Shows ROUGE, BM25, diversity scores for verbose mode
# NLP concept: transparent evaluation pipeline

def show_nlp_analysis(text, response, lang, verbose=True):
    """Print NLP feature analysis in verbose mode."""
    if not verbose: return
    features = extract_nlp_features(text)
    bm25_sc  = bm25_keyword_overlap(text, response)
    div_sc   = ngram_diversity(response)
    tfidf_sc = cosine_tfidf_score(text, response)
    print(f"  [NLP] Keywords   : {features['keywords'][:5]}")
    print(f"  [NLP] Entities   : {features['entities'][:3]}")
    print(f"  [NLP] BM25 overlap: {bm25_sc:.2f} | Diversity: {div_sc:.2f} | TF-IDF: {tfidf_sc:.2f}")


In [58]:
# ============================================================
# -- Step 8 : Main Chatbot with score-based backfill routing
# ============================================================
sarvam_history = []
llama_history  = []
gemini_history = []

_stats = {
    "mt5":    0,
    "sarvam": 0,
    "gemma":  0,
    "gemini": 0,
    "failed": 0,
}

@functools.lru_cache(maxsize=128)
def _cached_mt5(text, lang):
    return ask_mt5(text, lang)

def respond(user_text, verbose=False):
    lang  = detect_language(user_text)
    label = LANG_LABELS.get(lang, lang)
    if verbose: print(f"  [Detected: {label}]")

    if verbose: print("  [Routing to mT5-Large (primary) ...]")
    mt5_reply, mt5_score = _cached_mt5(user_text, lang)

    if mt5_reply and mt5_score >= MT5_SCORE_THRESHOLD:
        _stats["mt5"] += 1
        if verbose: show_nlp_analysis(user_text, mt5_reply, lang, verbose)
        return mt5_reply, f"mT5(score={mt5_score:.2f})", label

    if lang in INDIAN_LANGS:
        reply = ask_sarvam(user_text, lang, sarvam_history)
        if is_good_response(reply):
            _stats["sarvam"] += 1
            sarvam_history.append({"role":"user", "content":user_text})
            sarvam_history.append({"role":"assistant", "content":reply})
            return reply, "Sarvam(backfill)", label

        reply = ask_gemini(user_text, lang, gemini_history)
        if is_good_response(reply):
            _stats["gemini"] += 1
            gemini_history.append({"role": "user",  "parts": [{"text": user_text}]})
            gemini_history.append({"role": "model", "parts": [{"text": reply}]})
            return reply, "Gemini(backfill)", label
    else:
        reply = ask_llama(user_text, lang, llama_history)
        if is_good_response(reply):
            _stats["gemma"] += 1
            llama_history.append({"role":"user", "content":user_text})
            llama_history.append({"role":"assistant", "content":reply})
            return reply, "Gemma(backfill)", label

        reply = ask_gemini(user_text, lang, gemini_history)
        if is_good_response(reply):
            _stats["gemini"] += 1
            gemini_history.append({"role": "user",  "parts": [{"text": user_text}]})
            gemini_history.append({"role": "model", "parts": [{"text": reply}]})
            return reply, "Gemini(backfill)", label

    _stats["failed"] += 1
    return "Sorry, I could not generate a response.", "error", label

def print_stats():
    total = sum(_stats.values())
    print(f"\n-- Session stats --\nTotal: {total}\nGemini: {_stats['gemini']}")

In [82]:
# Final verification of the Gemini backfill with corrected model string
print("Verifying Gemini Backfill...")
test_query = "Explain why machine learning is useful."

# Using verbose=True to confirm routing avoiding the 404 error
reply, source, label = respond(test_query, verbose=True)

print(f"\nResult:")
print(f"Query  : {test_query}")
print(f"Source : {source}")
print(f"Reply  : {reply}")

Verifying Gemini Backfill...
  [Detected: English]
  [Routing to mT5-Large (primary) ...]

Result:
Query  : Explain why machine learning is useful.
Source : Gemini(backfill)
Reply  : Machine Learning (ML) is incredibly useful because it allows computers to find patterns, make decisions, and


In [92]:
# ── Chatbot loop (Testing Sarvam Hallucination Filter) ─────────
print("=" * 62)
print("  Multilingual Chatbot v13 (Filter Active)")
print("=" * 62)
print(f"  mT5-Large  (primary)  : {'READY' if MT5_READY else 'NOT LOADED'}")
print(f"  Sarvam-2B  (backfill) : {'READY' if SARVAM_READY else 'NOT LOADED'}")
print(f"  Gemini     (backfill) : {'READY' if GEMINI_READY else 'NOT SET UP'}")
print(f"  mT5 score threshold   : {MT5_SCORE_THRESHOLD}")
print("=" * 62)
print("  Languages : English | Hindi | Kannada | Hinglish | Kannada-EN")
print("  Commands  : stats | clear | verbose on/off | exit")
print()

VERBOSE = False

while True:
    try:
        user_input = input("You: ").strip()
    except EOFError:
        break
    if not user_input: continue
    low = user_input.lower()

    if low == "exit":
        print_stats()
        print("Chatbot: Goodbye!")
        break
    if low == "stats":        print_stats(); continue
    if low == "clear":
        sarvam_history.clear(); llama_history.clear(); gemini_history.clear()
        _cached_mt5.cache_clear()
        print("[Memory cleared.]\n"); continue
    if low == "verbose on":   VERBOSE = True;  print("[Verbose: ON]\n");  continue
    if low == "verbose off":  VERBOSE = False; print("[Verbose: OFF]\n"); continue

    reply, source, label = respond(user_input, verbose=VERBOSE)
    print(f"[{label}][{source}] Chatbot: {reply}\n")

  Multilingual Chatbot v13 (Filter Active)
  mT5-Large  (primary)  : READY
  Sarvam-2B  (backfill) : READY
  Gemini     (backfill) : READY
  mT5 score threshold   : 0.9
  Languages : English | Hindi | Kannada | Hinglish | Kannada-EN
  Commands  : stats | clear | verbose on/off | exit

You: NLP enu?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Kannada-English][Gemini(backfill)] Chatbot: **NLP** andre **Natural Language Processing**. Idhu **Artificial Intelligence (AI)** nalli thumba

You: exit

-- Session stats --
Total: 19
Gemini: 7
Chatbot: Goodbye!


In [ ]:

## HuggingFace Login (optional)

# Run ONCE if you see 401 errors. google/mt5-large and google/gemma-2-2b-it
# are public -- no gate. Only needed for gated models added later.

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab Secrets (HF_TOKEN)")
except Exception:
    HF_TOKEN = ""   # paste token here if needed
    if HF_TOKEN: login(token=HF_TOKEN); print("Logged in via token")
    else: print("No token -- public models load without login.")

In [90]:
from huggingface_hub import login
from google.colab import userdata

try:
    # This assumes you have added 'HF_TOKEN' to your Colab Secrets (key icon on the left)
    token = userdata.get('HF_TOKEN')
    login(token=token)
    print("✓ Successfully logged in to Hugging Face.")
except Exception as e:
    print(f"✗ Could not log in: {e}")
    print("Please ensure you have added your token to Colab Secrets as 'HF_TOKEN'.")

✓ Successfully logged in to Hugging Face.
